# LAB DAY 19: Building a GraphRAG System with a Tech Company Corpus

This notebook orchestrates the full pipeline:
1. **Indexing** – Extract (subject, relation, object) triples from the corpus using GPT-4o-mini.
2. **Graph Building** – Construct a NetworkX knowledge graph + optional Neo4j + NodeRAG.
3. **Flat RAG** – Embed corpus chunks into ChromaDB; retrieve by cosine similarity.
4. **GraphRAG** – Entity extraction → fuzzy node matching → 2-hop BFS → subgraph textualization.
5. **Evaluation** – LLM-as-judge on 20 benchmark questions; compare accuracy, tokens, and latency.

> **Prerequisites:** Copy `.env.example` to `.env` and fill in your `OPENAI_API_KEY`.

In [ ]:
import sys
import os
from pathlib import Path

# Ensure the project root is on the Python path
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

print(f"Project root: {PROJECT_ROOT}")
print(f"OpenAI key set: {'OPENAI_API_KEY' in os.environ and bool(os.environ['OPENAI_API_KEY'])}")

## Step 1: Corpus Inspection

The corpus consists of 20 paragraphs covering major AI companies. Let's preview it.

In [ ]:
corpus_path = PROJECT_ROOT / "data" / "tech_company_corpus.txt"
paragraphs = [p.strip() for p in corpus_path.read_text().split("\n\n") if p.strip()]
print(f"Total paragraphs: {len(paragraphs)}\n")
for i, p in enumerate(paragraphs[:3]):
    print(f"--- Paragraph {i} ---")
    print(p[:200])
    print()

## Step 2: Triple Extraction (Indexing)

We call `gpt-4o-mini` for each paragraph to extract (subject, relation, object) triples.
Entity names are normalized and deduplicated using fuzzy matching (threshold ≥ 90).

In [ ]:
from src.indexing import run_indexing

triples = run_indexing()
print(f"\nExtracted {len(triples)} unique triples.")
print("\nSample triples (first 10):")
for t in triples[:10]:
    print(f"  [{t['subject_type']}] {t['subject']} --[{t['relation']}]--> {t['object']} [{t['object_type']}]")

## Step 3: Knowledge Graph — NetworkX

Load triples into a `MultiDiGraph`, save as GraphML, and render a spring-layout visualization.

In [ ]:
from src.graph_networkx import run_graph_networkx

G = run_graph_networkx()
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

In [ ]:
# Display the graph screenshot inline
from IPython.display import Image, display

png_path = PROJECT_ROOT / "outputs" / "graph_screenshot.png"
if png_path.exists():
    display(Image(filename=str(png_path), width=1000))
else:
    print("Screenshot not found — run the cell above first.")

## Step 4: Neo4j (Optional)

If you have a running Neo4j instance, set credentials in `.env` and run the cell below.
It will MERGE all nodes and relationships, then print a Cypher query for the Browser.

In [ ]:
# Uncomment to run Neo4j loading
# from src.graph_neo4j import main as neo4j_main
# neo4j_main()

## Step 5: NodeRAG (Optional)

If the `NodeRAG` package is installed, this will build an all-in-one graph. Otherwise it skips gracefully.

In [ ]:
from src.graph_noderag import run_noderag
run_noderag()

## Step 6: Flat RAG — Build Index & Demo Query

Paragraphs are embedded with `text-embedding-3-small` and stored in ChromaDB.
At query time, the top-5 chunks by cosine similarity are stuffed into a QA prompt.

In [ ]:
from src.flat_rag import build_flat_index, query_flat_rag

collection = build_flat_index()
print(f"ChromaDB collection: {collection.count()} documents")

demo_q = "Who is the CEO of the company that acquired DeepMind?"
flat_result = query_flat_rag(demo_q, collection=collection)
print(f"\nQuestion: {demo_q}")
print(f"Flat RAG Answer: {flat_result['answer']}")
print(f"Tokens: {flat_result['tokens']} | Latency: {flat_result['latency_ms']:.0f} ms")

## Step 7: GraphRAG — Demo Query

The GraphRAG pipeline:
1. Extract entities from the question via LLM.
2. Fuzzy-match to graph nodes (threshold ≥ 70).
3. BFS up to 2 hops from matched nodes.
4. Textualize subgraph as numbered `<subject> --[relation]--> <object>` facts.
5. Feed into the same QA prompt as Flat RAG.

In [ ]:
from src.graph_rag import query_graph_rag

demo_q = "Who is the CEO of the company that acquired DeepMind?"
graph_result = query_graph_rag(demo_q)

print(f"Question: {demo_q}")
print(f"Matched nodes: {graph_result['matched_nodes']}")
print(f"\nSubgraph facts (first 15 lines):")
lines = graph_result['subgraph_text'].split('\n')[:15]
print('\n'.join(lines))
print(f"\nGraphRAG Answer: {graph_result['answer']}")
print(f"Tokens: {graph_result['tokens']} | Latency: {graph_result['latency_ms']:.0f} ms")

## Step 8: Full Evaluation — 20 Benchmark Questions

Run all 20 questions through both systems. An LLM judge compares each answer to the ground truth.
Results are saved to `outputs/benchmark_results.csv` and `outputs/cost_analysis.md`.

In [ ]:
from src.evaluate import main as eval_main
eval_main()

## Step 9: Results Visualization

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

csv_path = PROJECT_ROOT / "outputs" / "benchmark_results.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("GraphRAG vs Flat RAG — Benchmark Results", fontsize=14, fontweight="bold")
    
    # Accuracy by type
    types = df['type'].unique()
    flat_acc = [df[df['type']==t]['flat_correct'].mean() for t in types]
    graph_acc = [df[df['type']==t]['graph_correct'].mean() for t in types]
    x = range(len(types))
    axes[0].bar([i - 0.2 for i in x], flat_acc, 0.4, label='Flat RAG', color='#4A90D9')
    axes[0].bar([i + 0.2 for i in x], graph_acc, 0.4, label='GraphRAG', color='#E67E22')
    axes[0].set_xticks(list(x))
    axes[0].set_xticklabels(types)
    axes[0].set_title('Accuracy by Question Type')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_ylim(0, 1.1)
    axes[0].legend()
    
    # Avg tokens
    systems = ['Flat RAG', 'GraphRAG']
    avg_tokens = [df['flat_tokens'].mean(), df['graph_tokens'].mean()]
    axes[1].bar(systems, avg_tokens, color=['#4A90D9', '#E67E22'])
    axes[1].set_title('Avg Tokens per Question')
    axes[1].set_ylabel('Tokens')
    
    # Avg latency
    avg_latency = [df['flat_latency_ms'].mean(), df['graph_latency_ms'].mean()]
    axes[2].bar(systems, avg_latency, color=['#4A90D9', '#E67E22'])
    axes[2].set_title('Avg Latency per Question (ms)')
    axes[2].set_ylabel('Latency (ms)')
    
    plt.tight_layout()
    plt.savefig(str(PROJECT_ROOT / 'outputs' / 'benchmark_chart.png'), dpi=120, bbox_inches='tight')
    plt.show()
    
    print(f"\nOverall accuracy — Flat RAG: {df['flat_correct'].mean()*100:.1f}% | GraphRAG: {df['graph_correct'].mean()*100:.1f}%")
else:
    print("benchmark_results.csv not found — run evaluation first.")

## Step 10: Token Usage & Cost Summary

In [ ]:
from src.utils.llm_client import TRACKER

by_stage = TRACKER.tokens_by_stage()
print("Token Usage by Stage:")
print(f"{'Stage':<35} {'Prompt':>10} {'Completion':>12} {'Total':>10}")
print("-" * 70)
for stage, stats in sorted(by_stage.items()):
    print(f"{stage:<35} {stats['prompt']:>10,} {stats['completion']:>12,} {stats['total']:>10,}")

grand = TRACKER.total_tokens()
print("-" * 70)
print(f"{'TOTAL':<35} {'':>10} {'':>12} {grand:>10,}")

# Cost estimate
INPUT_PRICE = 0.150 / 1e6
OUTPUT_PRICE = 0.600 / 1e6
total_cost = sum(
    s['prompt'] * INPUT_PRICE + s['completion'] * OUTPUT_PRICE
    for s in by_stage.values()
)
print(f"\nEstimated total cost: ${total_cost:.4f} USD")

---
## Summary

This notebook demonstrated the complete GraphRAG pipeline:
- **Indexing**: LLM-extracted knowledge graph triples with fuzzy deduplication
- **GraphRAG**: 2-hop BFS from fuzzy-matched question entities → precise, structured context
- **Flat RAG**: Embedding-based top-5 chunk retrieval → dense context

**Key finding**: GraphRAG excels at **multi-hop questions** where the answer requires
traversing 2+ relationships (e.g., "Who is the CEO of the company that acquired X?").
Flat RAG performs comparably on single-hop questions but struggles to connect entities
across paragraph boundaries. Both systems correctly refuse to answer trick questions
about information not in the corpus.